In [1]:
import json
import os
from tqdm import tqdm

In [2]:
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

In [3]:
total_results = {}

In [4]:
import re

def get_number_from_string(text):
    # Tìm dấu gạch dưới và các chữ số ở cuối chuỗi
    match = re.search(r'_(\d+)$', text)
    if match:
        return int(match.group(1)) # Trả về số dưới dạng string
    return None

In [5]:
CACHE_URL_PATH = r"data\cache\mapping\hash2url.json"
CACHE_SEARCH_PATH = r"data\cache\mapping\hash_id_IR_20.json"
METADATA_PATH = r"data\metadata_medhop.json"

BASE_1000 = "data/ALL_ROUND/medhop_results_gg_direct"
DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"

In [6]:
def flatten_wiki_cache(file_path):
    # 1. Load dữ liệu từ file
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
    except Exception as e:
        print(f"Lỗi khi đọc file: {e}")
        return {}

    final_cache = {}

    for main_id, tasks in raw_data.items():
        for task_name, items in tasks.items():
            for hash_id, url in items.items():
                final_cache[hash_id] = url

    return final_cache

FILE_PATH = r'data\cache\all_hash_id_crawl_wiki_embed.json' 
url_mapping = flatten_wiki_cache(FILE_PATH)

print(f"Tổng số HashID thu được: {len(url_mapping)}")

Tổng số HashID thu được: 4074


In [7]:
CACHE_URL_WIKI = load_json(CACHE_URL_PATH)
CACHE_SEARCH = load_json(CACHE_SEARCH_PATH)
metadata = load_json(METADATA_PATH)
    
CACHE_URL_WIKI.update(url_mapping)

# SEARCH_MAPPING = load_json(r'data\cache\mapping\hash_id copy.json')
# CACHE_SEARCH.update(SEARCH_MAPPING)
# CACHE_SEARCH.update(load_json(r'data\cache\mapping\hash_id.json'))

# save_json(CACHE_SEARCH, CACHE_SEARCH_PATH)

In [8]:
len(CACHE_URL_WIKI)

53017

In [9]:
import re

def load_retrieval_map(data_input):
    """
    Map ID (WIKI_10, CRAWL_NER_WIKI_2,...) trực tiếp với hash_id theo thứ tự 1-1.
    """
    result_map = {}
    
    for key, entry in data_input.items():
        crawl_docs = entry.get("crawl_doc", [])
        hash_ids = entry.get("hash_id", [])
        for i, doc_text in enumerate(crawl_docs):
            for c in range(1, 50):
                s = "CRAWL_WIKI_{c}"
                if f"[{s}]" in doc_text:
                    result_map[s] = hash_ids[i]
                    break
                    
                s_2 = f"CRAWL_NER_WIKI_{c}" #CRAWL_NER_WIKI_8
                
                if f"[{s_2}]" in doc_text:
                    result_map[s_2] = hash_ids[i]
                    break
           
    return result_map

In [10]:
def extract_detail_regex(text):
    pattern = r'(?i)Detailed explanation[:\s]*(.*)'
    match = re.search(pattern, text)
    
    if match:
        return match.group(1).strip()
    return ""

In [11]:
def get_GG2URL(search_kw, search_kw_raw):
    dict_gg_id = {}
    if search_kw in CACHE_SEARCH or search_kw_raw in CACHE_SEARCH:
        hash_id_gg = CACHE_SEARCH.get(search_kw) or CACHE_SEARCH.get(search_kw_raw)
        if '.json' not in hash_id_gg:
            hash_id_gg = f"{hash_id_gg}.json"
        file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
        gg_data = load_json(file_gg)['google']['search_results']
        
        for idx, item in enumerate(gg_data, 1):
            dict_gg_id[f"GG_{idx}"] = item['link']
        
    return dict_gg_id        

In [12]:
def extract_detail_regex(text):
    pattern = r'(?i)Detailed explanation[:\s]*'
    match = re.search(pattern, text)
    
    if match:
        return text[match.start():]  # Lấy từ vị trí bắt đầu "Detailed explanation" đến cuối
    return text  # Trả về text gốc nếu không tìm thấy

# DIRECT

In [13]:
BASE_1000 = "data/ALL_ROUND/1000/medhop_results_gg_direct/all_ans" # KHÔNG MAPPING FINAL ANSWER
# BASE_1000 = "D:/DSKTLab/27-02-2026-CODE/medhop_results_gg_direct_ct6_mapping/all_ans" # CÓ MAPPING FINAL ANSWER
DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"

cach_output = {}

In [14]:
output_direct = {}

for qidx in tqdm(list(metadata.keys())):
    item = metadata[qidx]
    
    if item['type_q'] != 'direct':
        continue
    
    output_direct[qidx] = {
        'QIDX': qidx        
    }
    
    file_path = f"{BASE_1000}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    final_answer = tmp_data['1']['0']['0']['final_answer']
    question = tmp_data['1']['0']['0']['question']
    output_direct[qidx]['Question'] = question
    search_kw = tmp_data['1']['0']['0']['search_keyword'].lower().strip()
    search_kw_raw = search_kw + " "
    short_answer = final_answer['short_answer']
    output_direct[qidx]['short_answer'] = short_answer
    long_answer = final_answer['long_answer']
    
    retrieval_information = tmp_data['1']['0']['0']['retrieval_information']
    
    if final_answer['selection_status'] != 'selected':
        list_references = "This answer, generated by a zero-shot LLM, is not explainable."
        cach_output[qidx] = {
            'src': 'zeroshot',
            'final_answer': final_answer
        }
    else:
        sources = final_answer['sources']
        cach_output[qidx] = {
            'src': 'uetquintet model',
            'final_answer': final_answer
        }
    
    dict_gg_id2URL = get_GG2URL(search_kw, search_kw_raw)
    dict_crawl_wiki_data = load_retrieval_map(retrieval_information)
    dict_results_src = {}

    for src in sources:
        if src not in long_answer:
            continue
        if src in dict_gg_id2URL:
            dict_results_src[src] = dict_gg_id2URL[src]
            
        if src in dict_crawl_wiki_data:
            wiki_hashid = dict_crawl_wiki_data[src]
            dict_results_src[src] = CACHE_URL_WIKI.get(wiki_hashid, "")
            
    list_references = ""

    for src, link in dict_results_src.items():
        list_references += f"[{src}] {link}\n"    
        
    detail_exaplanations = extract_detail_regex(long_answer).replace("Detailed explanation", "\nExplanation")
    
    
    #CONCLUSION
    output_direct[qidx]['long_answer'] = f"""Question: {question}\nAnswer: {short_answer}\nDetail explanations: {detail_exaplanations}\nReferences:\n{list_references}"""
    
            

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:12<00:00, 77.11it/s]


In [16]:
save_json(cach_output, 'direct_final_answer_1000.json')

In [15]:
print(output_direct["150243"]['long_answer'])

Question: What rare congenital multiple-anomaly syndrome is caused by a mutation in the SKI gene on chromosome 1 and is characterized by abnormally long, slender fingers referred to as 'spider fingers'?
Answer: shprintzen-goldberg syndrome
Detail explanations: 
Explanation: The context includes multiple references to Shprintzen-Goldberg syndrome, a rare congenital multiple-anomaly syndrome. It is known by other names such as marfanoid-craniosynostosis syndrome and is characterized by skeletal abnormalities including features like arachnodactyly (spider fingers). Although the text does not explicitly state the genetic mutation, Shprintzen-Goldberg syndrome is associated with a mutation in the SKI gene on chromosome 1.Reference: [GG_2], [GG_4]. 
Explanation: The context cites Shprintzen Goldberg syndrome in multiple references. This rare congenital multiple‐anomaly syndrome, known to be caused by a mutation in the SKI gene on chromosome 1, is characterized by connective tissue abnormalit

In [16]:
save_json(output_direct, 'data/EVIDENCE/reference_direct_1000.json')
print(len(output_direct.keys()))

704


In [17]:
for qidx in output_direct:
    if qidx in total_results:
        print("ERROR")
    else:
        total_results[qidx] = output_direct[qidx]

# SEQUENCE

In [18]:
output_seq = {}
BASE_SEQ = "SEQ_medhop_results/all_ans"
seq_combine = load_json(r"SEQ_medhop_results\seq_combine_zeroshot.json")

for qidx in tqdm(list(metadata.keys())):    
    item = metadata[qidx]
    if item['type_q'] != 'sequence':
        continue
    
    output_seq[qidx] = {}    
    
    file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    prev_answer = ""
    original_question = item['original question']
    output_seq[qidx]['QIDX'] = qidx
    output_seq[qidx]['Question'] = original_question
    
    output_seq[qidx]['long_answer'] = "Question: " + original_question + '\n\n' + "Follow the reasoning step by step:\n"
    
    for step, step_data in tmp_data['1']['0'].items():
        step = str(step)
        
        final_answer = tmp_data['1']['0'][step]['final_answer']
        question = tmp_data['1']['0'][step]['question']
        search_kw = tmp_data['1']['0'][step]['search_keyword']
                
        short_answer = final_answer['short_answer']
        long_answer = final_answer['long_answer']
        
        if step != '0':
            search_kw = search_kw + " " + prev_answer
            
        search_kw = search_kw.lower().strip()
        search_kw_raw = search_kw + " "
        prev_answer = short_answer.lower()
      
        if final_answer['selection_status'] != 'selected':
            list_references = "This answer, generated by a zero-shot LLM, is not explainable."
        else:
            sources = final_answer['sources']
        
        retrieval_information = tmp_data['1']['0'][step]['retrieval_information']
        dict_gg_id2URL = get_GG2URL(search_kw, search_kw_raw)
        dict_crawl_wiki_data = load_retrieval_map(retrieval_information)
        dict_results_src = {}

        for src in sources:
            if src not in long_answer:
                continue
            if src in dict_gg_id2URL:
                dict_results_src[src] = dict_gg_id2URL[src]
                
            if src in dict_crawl_wiki_data:
                wiki_hashid = dict_crawl_wiki_data[src]
                dict_results_src[src] = CACHE_URL_WIKI.get(wiki_hashid, "")
                
        list_references = ""

        for src, link in dict_results_src.items():
            list_references += f"[{src}] {link}\n"    
            
        detail_exaplanations = extract_detail_regex(long_answer).replace("Detailed explanation", "\nExplanation")
        
        
        output_seq[qidx]['long_answer'] += f"""Find answer for subquestion {int(step)+1}: {question}\nAnswer: {short_answer}\nDetail explanations: {detail_exaplanations}\nReferences:\n{list_references}\n"""
        
    #CONCLUSION
    conclusion_ans = seq_combine[qidx]
    output_seq[qidx]['long_answer'] += "\nConclusion:\n" + f"Answer: {conclusion_ans['short_answer']}\n" + f"Explanation: {conclusion_ans['long_answer']}"
    output_seq[qidx]['short_answer'] = conclusion_ans['short_answer']


100%|██████████| 1000/1000 [00:04<00:00, 220.47it/s]


In [19]:
print(len(output_seq.keys()))
save_json(output_seq, 'data/EVIDENCE/reference_sequence_1000.json')

225


In [20]:
for qidx in output_seq:
    if qidx in total_results:
        print("ERROR")
    else:
        total_results[qidx] = output_seq[qidx]

In [21]:
print(output_seq['102914']['long_answer'])

Question: Is Peutz–Jeghers syndrome a genetic disorder associated with gene variants on non-sex chromosomes?

Follow the reasoning step by step:
Find answer for subquestion 1: What is Peutz–Jeghers syndrome and is it classified as a genetic disorder?
Answer: yes
Detail explanations: 
Explanation: The context explicitly describes Peutz–Jeghers syndrome as an autosomal dominant genetic disorder. It is characterized by the development of hamartomatous polyps throughout the gastrointestinal tract and mucocutaneous pigmentation, and is associated with mutations (e.g., in the STK11 gene) that confer a genetic predisposition to other complications such as cancer. This confirms that Peutz–Jeghers syndrome is indeed classified as a genetic disorder.
References:
[GG_15] https://www.genomicseducation.hee.nhs.uk/documents/peutz-jeghers-syndrome/
[GG_18] https://www.ebsco.com/research-starters/health-and-medicine/peutz-jeghers-syndrome-pjs
[GG_12] https://my.clevelandclinic.org/health/diseases/1736

# PARALLEL

In [22]:
output_parallel = {}
BASE_PAR = "data/ALL_ROUND/1000/medhop_results_ct6_r36_parallel_seq/all_ans"
parallel_combine = load_json(r'data\ALL_ROUND\1000\medhop_results_ct6_r36_parallel_seq\parallel_combine_zeroshot_inf.json')

for qidx in tqdm(list(metadata.keys())):    
    item = metadata[qidx]
       
    if item['type_q'] != 'parallel':
        continue
    output_parallel[qidx] = {'QIDX': qidx}
    
    file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"

    tmp_data = load_json(file_path)
    original_question = item['original question']
    output_parallel[qidx]['Question'] = original_question

    output_parallel[qidx]['long_answer'] = f"Question: {original_question}\n\nFollow the independent reasoning steps:\n"
    for qi in ["1", "2", "3", "4"]:
        if qi not in tmp_data:
            break      
        
        for step, step_data in tmp_data[qi]['0'].items():
            step = str(step)
            final_answer = tmp_data[qi]['0'][step]['final_answer']
            question = tmp_data[qi]['0'][step]['question']
            search_kw = tmp_data[qi]['0'][step]['search_keyword'].lower().strip()
            
            short_answer = final_answer['short_answer']
            long_answer = final_answer['long_answer']
           
            search_kw_raw = search_kw + " "            
            
            if final_answer['selection_status'] != 'selected':
                list_references = "This answer, generated by a zero-shot LLM, is not explainable."
            else:
                sources = final_answer['sources']
                
            retrieval_information = tmp_data[qi]['0'][step]['retrieval_information']
            dict_gg_id2URL = get_GG2URL(search_kw, search_kw_raw)
            dict_crawl_wiki_data = load_retrieval_map(retrieval_information)
            dict_results_src = {}

            for src in sources:
                if src not in long_answer:
                    continue
                if src in dict_gg_id2URL:
                    dict_results_src[src] = dict_gg_id2URL[src]
                    
                if src in dict_crawl_wiki_data:
                    wiki_hashid = dict_crawl_wiki_data[src]
                    dict_results_src[src] = CACHE_URL_WIKI.get(wiki_hashid, "")
                    
            list_references = ""

            for src, link in dict_results_src.items():
                list_references += f"[{src}] {link}\n"    
                
            detail_exaplanations = extract_detail_regex(long_answer).replace("Detailed explanation", "\nExplanation")

            output_parallel[qidx]['long_answer'] += f"""Find answer for question: {question}\nAnswer: {short_answer}\nDetail explanations: {detail_exaplanations}\nReferences:\n{list_references}\n"""
        
    #CONCLUSION
    conclusion_ans = parallel_combine[qidx]
    output_parallel[qidx]['long_answer'] += "\nConclusion:\n" + f"Answer: {conclusion_ans['short_answer']}\n" + f"Explanation: {conclusion_ans['long_answer']}"
    output_parallel[qidx]['short_answer'] = conclusion_ans['short_answer']

100%|██████████| 1000/1000 [00:01<00:00, 611.19it/s]


In [23]:
save_json(output_parallel, 'data/EVIDENCE/reference_parallel_1000.json')

In [24]:
for qidx in output_parallel:
    if qidx in total_results:
        print("ERROR")
    else:
        total_results[qidx] = output_parallel[qidx]

In [25]:
print(output_parallel['196089']['long_answer'])

Question: Are Absent tibia-polydactyly-arachnoid cyst syndrome and Charcot-Marie-Tooth disease caused by the same chromosomal abnormalities?

Follow the independent reasoning steps:
Find answer for question: What chromosomal abnormalities are associated with Absent tibia-polydactyly-arachnoid cyst syndrome?
Answer: yes
Detail explanations: Several patients who fulfil the clinical triad of absent/hypoplastic tibia, post-axial polydactyly and large arachnoid (or interhemispheric) cysts have been shown to carry pathogenic chromosomal imbalances. Most commonly the phenotype occurs in the setting of a distal 8q24.3 microdeletion (Verheij syndrome) that removes the PUF60-SCRIB critical region; in other affected individuals interstitial deletions of 2q31 involving the HOXD gene cluster or more proximal 6q15-q16 deletions have been documented. These data demonstrate that specific chromosomal abnormalities are indeed associated with Absent tibia-polydactyly-arachnoid cyst syndrome.
References:


In [26]:
import pandas as pd

data_list = []
for qidx, info in total_results.items():
    if 'Question' not in total_results[qidx]:
        print(qidx)
        
    data_list.append({
        'QIDX': qidx,
        'Question': info['Question'],
        'Short Answer': info['short_answer'],
        'Long Answer': info['long_answer']
    })

df = pd.DataFrame(data_list)

# Xuất ra CSV (sử dụng utf-8-sig để mở bằng Excel không bị lỗi font tiếng Việt)
df.to_csv('data/SUBMIT/submission_1000q.csv', index=False, encoding='utf-8-sig')

print("Đã lưu file CSV thành công!")

Đã lưu file CSV thành công!
